In [ ]:
pip install flask requests pandas yfinance plotly

In [ ]:
import os
import time
import math
import requests
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go

from flask import Flask, request, render_template_string


app = Flask(__name__)

# SEC asks API users to identify themselves with a descriptive User-Agent.
# Set your email in the environment if you want:
#   export SEC_USER_AGENT="Your Name your.email@example.com"
SEC_USER_AGENT = os.getenv(
    "SEC_USER_AGENT",
    "Fundamental Analysis Dashboard contact@example.com"
)

SEC_HEADERS = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

SEC_ARCHIVE_HEADERS = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate"
}

REQUEST_DELAY_SECONDS = 0.12

SEC_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_SUBMISSIONS_URL = "https://data.sec.gov/submissions/CIK{cik}.json"
SEC_COMPANY_FACTS_URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"


# ----------------------------
# Basic helpers
# ----------------------------

def safe_float(x):
    try:
        if x is None:
            return None
        value = float(x)
        if math.isnan(value) or math.isinf(value):
            return None
        return value
    except Exception:
        return None


def fmt_num(x, decimals=2):
    x = safe_float(x)
    if x is None:
        return "N/A"

    abs_x = abs(x)
    if abs_x >= 1_000_000_000_000:
        return f"${x / 1_000_000_000_000:.{decimals}f}T"
    if abs_x >= 1_000_000_000:
        return f"${x / 1_000_000_000:.{decimals}f}B"
    if abs_x >= 1_000_000:
        return f"${x / 1_000_000:.{decimals}f}M"
    if abs_x >= 1_000:
        return f"${x / 1_000:.{decimals}f}K"
    return f"${x:.{decimals}f}"


def fmt_plain(x, decimals=2):
    x = safe_float(x)
    if x is None:
        return "N/A"

    abs_x = abs(x)
    if abs_x >= 1_000_000_000_000:
        return f"{x / 1_000_000_000_000:.{decimals}f}T"
    if abs_x >= 1_000_000_000:
        return f"{x / 1_000_000_000:.{decimals}f}B"
    if abs_x >= 1_000_000:
        return f"{x / 1_000_000:.{decimals}f}M"
    if abs_x >= 1_000:
        return f"{x / 1_000:.{decimals}f}K"
    return f"{x:.{decimals}f}"


def fmt_ratio(x, decimals=2):
    x = safe_float(x)
    if x is None:
        return "N/A"
    return f"{x:.{decimals}f}"


def fmt_pct(x, decimals=1):
    x = safe_float(x)
    if x is None:
        return "N/A"
    return f"{x * 100:.{decimals}f}%"


def divide(a, b):
    a = safe_float(a)
    b = safe_float(b)
    if a is None or b is None or b == 0:
        return None
    return a / b


def sec_get_json(url, headers=None):
    headers = headers or SEC_HEADERS
    time.sleep(REQUEST_DELAY_SECONDS)

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()


# ----------------------------
# SEC data retrieval
# ----------------------------

def get_cik_from_ticker(ticker):
    ticker = ticker.upper().strip()

    data = sec_get_json(SEC_TICKERS_URL, headers=SEC_ARCHIVE_HEADERS)

    for _, company in data.items():
        if company.get("ticker", "").upper() == ticker:
            cik = str(company["cik_str"]).zfill(10)
            return {
                "ticker": ticker,
                "cik": cik,
                "company_name": company.get("title")
            }

    raise ValueError(f"Could not find CIK for ticker: {ticker}")


def get_company_profile(cik):
    url = SEC_SUBMISSIONS_URL.format(cik=cik)
    return sec_get_json(url)


def get_company_facts(cik):
    url = SEC_COMPANY_FACTS_URL.format(cik=cik)
    return sec_get_json(url)


# ----------------------------
# SEC facts parsing
# ----------------------------

def get_us_gaap_facts(company_facts):
    return company_facts.get("facts", {}).get("us-gaap", {})


def extract_fact_df(company_facts, tag, unit_preference=None):
    """
    Returns a normalized DataFrame for a specific us-gaap tag.
    """
    facts = get_us_gaap_facts(company_facts)

    if tag not in facts:
        return pd.DataFrame()

    units = facts[tag].get("units", {})
    if not units:
        return pd.DataFrame()

    selected_unit = None

    if unit_preference and unit_preference in units:
        selected_unit = unit_preference
    elif "USD" in units:
        selected_unit = "USD"
    elif "shares" in units:
        selected_unit = "shares"
    elif "USD/shares" in units:
        selected_unit = "USD/shares"
    else:
        selected_unit = list(units.keys())[0]

    rows = units.get(selected_unit, [])
    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    for col in ["val", "fy", "fp"]:
        if col not in df.columns:
            df[col] = None

    if "end" in df.columns:
        df["end"] = pd.to_datetime(df["end"], errors="coerce")
    else:
        df["end"] = pd.NaT

    if "start" in df.columns:
        df["start"] = pd.to_datetime(df["start"], errors="coerce")
    else:
        df["start"] = pd.NaT

    df["val"] = pd.to_numeric(df["val"], errors="coerce")
    df["tag"] = tag
    df["unit"] = selected_unit

    # Remove amendments/duplicates by keeping latest filed for same period if possible.
    if "filed" in df.columns:
        df["filed"] = pd.to_datetime(df["filed"], errors="coerce")
        sort_cols = ["end", "filed"]
    else:
        sort_cols = ["end"]

    df = df.sort_values(sort_cols).dropna(subset=["end", "val"])

    return df


def first_available_fact(company_facts, tags, unit_preference=None):
    """
    Returns the first non-empty fact DataFrame among candidate tags.
    """
    for tag in tags:
        df = extract_fact_df(company_facts, tag, unit_preference=unit_preference)
        if not df.empty:
            return tag, df
    return None, pd.DataFrame()


def latest_instant_value(company_facts, tags, unit_preference="USD"):
    """
    Best for balance sheet / point-in-time facts.
    """
    tag, df = first_available_fact(company_facts, tags, unit_preference=unit_preference)
    if df.empty:
        return None, tag

    # Prefer annual or quarterly forms if available.
    if "form" in df.columns:
        preferred = df[df["form"].isin(["10-K", "10-Q", "20-F", "40-F"])]
        if not preferred.empty:
            df = preferred

    row = df.sort_values("end").iloc[-1]
    return safe_float(row["val"]), tag


def annual_series(company_facts, tags, unit_preference="USD"):
    """
    Returns annual series, usually from 10-K / FY rows.
    """
    tag, df = first_available_fact(company_facts, tags, unit_preference=unit_preference)
    if df.empty:
        return tag, pd.DataFrame()

    filtered = df.copy()

    if "form" in filtered.columns:
        form_filtered = filtered[filtered["form"].isin(["10-K", "20-F", "40-F"])]
        if not form_filtered.empty:
            filtered = form_filtered

    if "fp" in filtered.columns:
        fy_filtered = filtered[filtered["fp"].astype(str).str.upper().eq("FY")]
        if not fy_filtered.empty:
            filtered = fy_filtered

    # Keep one value per fiscal year.
    if "fy" in filtered.columns:
        filtered = (
            filtered
            .dropna(subset=["fy"])
            .sort_values(["fy", "end"])
            .drop_duplicates(subset=["fy"], keep="last")
        )
        filtered["period"] = filtered["fy"].astype(int).astype(str)
    else:
        filtered["period"] = filtered["end"].dt.year.astype(str)

    return tag, filtered.sort_values("end")


def ttm_value(company_facts, tags, unit_preference="USD"):
    """
    Approximate trailing-twelve-month value by summing latest 4 quarterly facts
    when available. Falls back to latest annual value.
    """
    tag, df = first_available_fact(company_facts, tags, unit_preference=unit_preference)
    if df.empty:
        return None, tag

    quarterly = df.copy()

    if "form" in quarterly.columns:
        quarterly = quarterly[quarterly["form"].isin(["10-Q", "10-K"])]

    if "fp" in quarterly.columns:
        q_filtered = quarterly[quarterly["fp"].astype(str).str.upper().isin(["Q1", "Q2", "Q3", "Q4"])]
        if not q_filtered.empty:
            quarterly = q_filtered

    # Quarterly duration facts normally have start/end dates.
    quarterly = quarterly.dropna(subset=["start", "end", "val"])
    if not quarterly.empty:
        quarterly["days"] = (quarterly["end"] - quarterly["start"]).dt.days

        # Prefer roughly quarterly duration rows.
        quarterly_like = quarterly[(quarterly["days"] >= 60) & (quarterly["days"] <= 120)]

        if len(quarterly_like) >= 4:
            latest_four = quarterly_like.sort_values("end").tail(4)
            return safe_float(latest_four["val"].sum()), tag

    # Fallback to annual.
    _, annual = annual_series(company_facts, tags, unit_preference=unit_preference)
    if not annual.empty:
        return safe_float(annual.sort_values("end").iloc[-1]["val"]), tag

    return None, tag


# ----------------------------
# Financial data calculation
# ----------------------------

FUNDAMENTAL_TAGS = {
    "revenue": [
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues",
        "SalesRevenueNet"
    ],
    "gross_profit": [
        "GrossProfit"
    ],
    "operating_income": [
        "OperatingIncomeLoss"
    ],
    "net_income": [
        "NetIncomeLoss",
        "ProfitLoss"
    ],
    "eps_diluted": [
        "EarningsPerShareDiluted",
        "IncomeLossFromContinuingOperationsPerDilutedShare"
    ],
    "assets": [
        "Assets"
    ],
    "liabilities": [
        "Liabilities"
    ],
    "equity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
        "CommonStocksIncludingAdditionalPaidInCapital"
    ],
    "cash": [
        "CashAndCashEquivalentsAtCarryingValue",
        "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"
    ],
    "operating_cash_flow": [
        "NetCashProvidedByUsedInOperatingActivities",
        "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"
    ],
    "capex": [
        "PaymentsToAcquirePropertyPlantAndEquipment",
        "PaymentsToAcquireProductiveAssets"
    ],
    "shares_diluted": [
        "WeightedAverageNumberOfDilutedSharesOutstanding",
        "WeightedAverageNumberOfShareOutstandingBasicAndDiluted",
        "WeightedAverageNumberOfSharesOutstandingDiluted"
    ]
}


def get_yahoo_data(ticker):
    stock = yf.Ticker(ticker)

    info = {}
    try:
        info = stock.info or {}
    except Exception:
        info = {}

    hist = stock.history(period="2y", interval="1d", auto_adjust=True)

    current_price = None
    if not hist.empty:
        current_price = safe_float(hist["Close"].dropna().iloc[-1])

    yahoo_market_cap = safe_float(info.get("marketCap"))
    yahoo_shares = safe_float(info.get("sharesOutstanding"))
    yahoo_name = info.get("longName") or info.get("shortName")

    return {
        "info": info,
        "history": hist,
        "current_price": current_price,
        "market_cap": yahoo_market_cap,
        "shares_outstanding": yahoo_shares,
        "company_name": yahoo_name
    }


def build_price_chart(history, ticker):
    if history is None or history.empty:
        return ""

    df = history.copy().reset_index()

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df["Date"],
            y=df["Close"],
            mode="lines",
            name="Close"
        )
    )

    fig.update_layout(
        title=f"{ticker.upper()} Price History",
        xaxis_title="Date",
        yaxis_title="Adjusted Close",
        height=420,
        margin=dict(l=40, r=30, t=60, b=40)
    )

    return fig.to_html(full_html=False, include_plotlyjs="cdn")


def build_bar_chart(series_df, title, y_title):
    if series_df is None or series_df.empty:
        return ""

    plot_df = series_df.tail(6).copy()

    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=plot_df["period"],
            y=plot_df["val"],
            name=title
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title="Fiscal Year",
        yaxis_title=y_title,
        height=380,
        margin=dict(l=40, r=30, t=60, b=40)
    )

    return fig.to_html(full_html=False, include_plotlyjs=False)


def analyze_stock(ticker):
    ticker = ticker.upper().strip()

    cik_data = get_cik_from_ticker(ticker)
    cik = cik_data["cik"]

    profile = get_company_profile(cik)
    company_facts = get_company_facts(cik)
    yahoo = get_yahoo_data(ticker)

    company_name = (
        yahoo.get("company_name")
        or profile.get("name")
        or cik_data.get("company_name")
        or ticker
    )

    current_price = yahoo["current_price"]

    revenue_ttm, revenue_tag = ttm_value(company_facts, FUNDAMENTAL_TAGS["revenue"])
    gross_profit_ttm, gross_profit_tag = ttm_value(company_facts, FUNDAMENTAL_TAGS["gross_profit"])
    operating_income_ttm, operating_income_tag = ttm_value(company_facts, FUNDAMENTAL_TAGS["operating_income"])
    net_income_ttm, net_income_tag = ttm_value(company_facts, FUNDAMENTAL_TAGS["net_income"])
    eps_diluted_ttm, eps_tag = ttm_value(company_facts, FUNDAMENTAL_TAGS["eps_diluted"], unit_preference="USD/shares")
    ocf_ttm, ocf_tag = ttm_value(company_facts, FUNDAMENTAL_TAGS["operating_cash_flow"])
    capex_ttm, capex_tag = ttm_value(company_facts, FUNDAMENTAL_TAGS["capex"])
    shares_diluted_ttm, shares_tag = ttm_value(company_facts, FUNDAMENTAL_TAGS["shares_diluted"], unit_preference="shares")

    assets, assets_tag = latest_instant_value(company_facts, FUNDAMENTAL_TAGS["assets"])
    liabilities, liabilities_tag = latest_instant_value(company_facts, FUNDAMENTAL_TAGS["liabilities"])
    equity, equity_tag = latest_instant_value(company_facts, FUNDAMENTAL_TAGS["equity"])
    cash, cash_tag = latest_instant_value(company_facts, FUNDAMENTAL_TAGS["cash"])

    shares_for_market_cap = yahoo.get("shares_outstanding") or shares_diluted_ttm

    market_cap = yahoo.get("market_cap")
    if market_cap is None and current_price is not None and shares_for_market_cap is not None:
        market_cap = current_price * shares_for_market_cap

    # SEC capex values are often positive cash outflows. Some companies report negative.
    # Normalize FCF as OCF - absolute capex if capex appears positive.
    free_cash_flow = None
    if ocf_ttm is not None and capex_ttm is not None:
        free_cash_flow = ocf_ttm - abs(capex_ttm)

    pe = divide(current_price, eps_diluted_ttm)
    ps = divide(market_cap, revenue_ttm)
    pb = divide(market_cap, equity)
    debt_to_equity = divide(liabilities, equity)
    gross_margin = divide(gross_profit_ttm, revenue_ttm)
    operating_margin = divide(operating_income_ttm, revenue_ttm)
    net_margin = divide(net_income_ttm, revenue_ttm)
    fcf_yield = divide(free_cash_flow, market_cap)

    revenue_tag_used, revenue_series = annual_series(company_facts, FUNDAMENTAL_TAGS["revenue"])
    net_income_tag_used, net_income_series = annual_series(company_facts, FUNDAMENTAL_TAGS["net_income"])
    fcf_tag_used, ocf_series = annual_series(company_facts, FUNDAMENTAL_TAGS["operating_cash_flow"])

    price_chart = build_price_chart(yahoo["history"], ticker)
    revenue_chart = build_bar_chart(revenue_series, "Annual Revenue", "Revenue")
    net_income_chart = build_bar_chart(net_income_series, "Annual Net Income", "Net Income")
    ocf_chart = build_bar_chart(ocf_series, "Annual Operating Cash Flow", "Operating Cash Flow")

    metrics = [
        {
            "label": "Current Price",
            "value": fmt_num(current_price),
            "raw": current_price,
            "note": "Yahoo Finance"
        },
        {
            "label": "Market Cap",
            "value": fmt_num(market_cap),
            "raw": market_cap,
            "note": "Yahoo Finance or price × shares"
        },
        {
            "label": "Revenue TTM",
            "value": fmt_num(revenue_ttm),
            "raw": revenue_ttm,
            "note": revenue_tag
        },
        {
            "label": "Net Income TTM",
            "value": fmt_num(net_income_ttm),
            "raw": net_income_ttm,
            "note": net_income_tag
        },
        {
            "label": "EPS Diluted TTM",
            "value": fmt_ratio(eps_diluted_ttm),
            "raw": eps_diluted_ttm,
            "note": eps_tag
        },
        {
            "label": "P/E",
            "value": fmt_ratio(pe),
            "raw": pe,
            "note": "Price / EPS diluted TTM"
        },
        {
            "label": "P/S",
            "value": fmt_ratio(ps),
            "raw": ps,
            "note": "Market cap / revenue TTM"
        },
        {
            "label": "P/B",
            "value": fmt_ratio(pb),
            "raw": pb,
            "note": "Market cap / equity"
        },
        {
            "label": "Debt/Equity",
            "value": fmt_ratio(debt_to_equity),
            "raw": debt_to_equity,
            "note": "Liabilities / equity"
        },
        {
            "label": "Free Cash Flow TTM",
            "value": fmt_num(free_cash_flow),
            "raw": free_cash_flow,
            "note": "Operating cash flow - capex"
        },
        {
            "label": "FCF Yield",
            "value": fmt_pct(fcf_yield),
            "raw": fcf_yield,
            "note": "FCF / market cap"
        },
        {
            "label": "Gross Margin",
            "value": fmt_pct(gross_margin),
            "raw": gross_margin,
            "note": "Gross profit / revenue"
        },
        {
            "label": "Operating Margin",
            "value": fmt_pct(operating_margin),
            "raw": operating_margin,
            "note": "Operating income / revenue"
        },
        {
            "label": "Net Margin",
            "value": fmt_pct(net_margin),
            "raw": net_margin,
            "note": "Net income / revenue"
        },
        {
            "label": "Cash",
            "value": fmt_num(cash),
            "raw": cash,
            "note": cash_tag
        },
        {
            "label": "Assets",
            "value": fmt_num(assets),
            "raw": assets,
            "note": assets_tag
        },
        {
            "label": "Liabilities",
            "value": fmt_num(liabilities),
            "raw": liabilities,
            "note": liabilities_tag
        },
        {
            "label": "Equity",
            "value": fmt_num(equity),
            "raw": equity,
            "note": equity_tag
        }
    ]

    return {
        "ticker": ticker,
        "cik": cik,
        "company_name": company_name,
        "sic": profile.get("sic"),
        "sic_description": profile.get("sicDescription"),
        "fiscal_year_end": profile.get("fiscalYearEnd"),
        "metrics": metrics,
        "price_chart": price_chart,
        "revenue_chart": revenue_chart,
        "net_income_chart": net_income_chart,
        "ocf_chart": ocf_chart,
        "facts_used": {
            "revenue": revenue_tag,
            "gross_profit": gross_profit_tag,
            "operating_income": operating_income_tag,
            "net_income": net_income_tag,
            "eps": eps_tag,
            "operating_cash_flow": ocf_tag,
            "capex": capex_tag,
            "shares": shares_tag,
            "assets": assets_tag,
            "liabilities": liabilities_tag,
            "equity": equity_tag,
            "cash": cash_tag
        }
    }


# ----------------------------
# Flask UI
# ----------------------------

HTML_TEMPLATE = """
<!doctype html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <title>Fundamental Analysis Dashboard</title>
    <meta name="viewport" content="width=device-width, initial-scale=1">

    <style>
        :root {
            --bg: #0f172a;
            --panel: #111827;
            --card: #1f2937;
            --muted: #9ca3af;
            --text: #f9fafb;
            --accent: #60a5fa;
            --good: #34d399;
            --warn: #fbbf24;
            --bad: #f87171;
            --border: #374151;
        }

        body {
            margin: 0;
            font-family: Arial, Helvetica, sans-serif;
            background: linear-gradient(135deg, #0f172a 0%, #111827 100%);
            color: var(--text);
        }

        .container {
            max-width: 1280px;
            margin: 0 auto;
            padding: 28px;
        }

        .hero {
            background: rgba(31, 41, 55, 0.9);
            border: 1px solid var(--border);
            border-radius: 22px;
            padding: 28px;
            box-shadow: 0 20px 50px rgba(0,0,0,0.25);
            margin-bottom: 24px;
        }

        h1 {
            margin: 0 0 8px;
            font-size: 34px;
            letter-spacing: -0.03em;
        }

        .subtitle {
            color: var(--muted);
            margin-bottom: 22px;
        }

        form {
            display: flex;
            gap: 12px;
            flex-wrap: wrap;
        }

        input[type="text"] {
            padding: 14px 16px;
            border-radius: 14px;
            border: 1px solid var(--border);
            background: #020617;
            color: var(--text);
            font-size: 16px;
            min-width: 240px;
        }

        button {
            padding: 14px 18px;
            border-radius: 14px;
            border: 0;
            background: var(--accent);
            color: #0f172a;
            font-weight: 700;
            cursor: pointer;
            font-size: 16px;
        }

        button:hover {
            opacity: 0.9;
        }

        .company {
            margin-top: 22px;
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
            gap: 12px;
        }

        .info-pill {
            background: #020617;
            border: 1px solid var(--border);
            border-radius: 14px;
            padding: 12px 14px;
        }

        .info-pill span {
            display: block;
            color: var(--muted);
            font-size: 12px;
            text-transform: uppercase;
            letter-spacing: 0.08em;
            margin-bottom: 4px;
        }

        .grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(230px, 1fr));
            gap: 16px;
            margin-bottom: 24px;
        }

        .card {
            background: rgba(31, 41, 55, 0.9);
            border: 1px solid var(--border);
            border-radius: 18px;
            padding: 18px;
            box-shadow: 0 12px 30px rgba(0,0,0,0.18);
        }

        .metric-label {
            color: var(--muted);
            font-size: 13px;
            margin-bottom: 8px;
        }

        .metric-value {
            font-size: 27px;
            font-weight: 800;
            letter-spacing: -0.03em;
        }

        .metric-note {
            margin-top: 8px;
            color: var(--muted);
            font-size: 12px;
            overflow-wrap: anywhere;
        }

        .chart {
            background: white;
            color: #111827;
            border-radius: 18px;
            padding: 8px;
            margin-bottom: 24px;
            overflow: hidden;
        }

        .chart-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(420px, 1fr));
            gap: 20px;
        }

        .error {
            background: rgba(248, 113, 113, 0.15);
            border: 1px solid rgba(248, 113, 113, 0.45);
            color: #fecaca;
            padding: 18px;
            border-radius: 16px;
            margin-bottom: 20px;
        }

        .small {
            color: var(--muted);
            font-size: 13px;
            line-height: 1.5;
        }

        .facts {
            margin-top: 20px;
            background: rgba(2, 6, 23, 0.8);
            border: 1px solid var(--border);
            border-radius: 18px;
            padding: 18px;
        }

        .facts code {
            color: #bfdbfe;
        }

        @media (max-width: 520px) {
            .container {
                padding: 16px;
            }

            h1 {
                font-size: 28px;
            }

            .chart-grid {
                grid-template-columns: 1fr;
            }
        }
    </style>
</head>

<body>
<div class="container">

    <section class="hero">
        <h1>Fundamental Analysis Dashboard</h1>
        <div class="subtitle">
            Pulls SEC EDGAR company facts and Yahoo Finance price data.
        </div>

        <form method="get">
            <input
                type="text"
                name="ticker"
                placeholder="Enter ticker, e.g. AAPL"
                value="{{ ticker or '' }}"
                required
            >
            <button type="submit">Analyze</button>
        </form>

        <p class="small">
            Educational use only. This dashboard does not make investment recommendations.
            SEC taxonomy tags can differ by company, so verify important metrics against filings.
        </p>

        {% if result %}
        <div class="company">
            <div class="info-pill">
                <span>Company</span>
                {{ result.company_name }}
            </div>
            <div class="info-pill">
                <span>Ticker</span>
                {{ result.ticker }}
            </div>
            <div class="info-pill">
                <span>CIK</span>
                {{ result.cik }}
            </div>
            <div class="info-pill">
                <span>Industry</span>
                {{ result.sic_description or 'N/A' }}
            </div>
            <div class="info-pill">
                <span>Fiscal Year End</span>
                {{ result.fiscal_year_end or 'N/A' }}
            </div>
        </div>
        {% endif %}
    </section>

    {% if error %}
    <div class="error">
        <strong>Error:</strong> {{ error }}
    </div>
    {% endif %}

    {% if result %}
    <section class="grid">
        {% for metric in result.metrics %}
        <div class="card">
            <div class="metric-label">{{ metric.label }}</div>
            <div class="metric-value">{{ metric.value }}</div>
            <div class="metric-note">{{ metric.note or 'N/A' }}</div>
        </div>
        {% endfor %}
    </section>

    {% if result.price_chart %}
    <section class="chart">
        {{ result.price_chart | safe }}
    </section>
    {% endif %}

    <section class="chart-grid">
        {% if result.revenue_chart %}
        <div class="chart">
            {{ result.revenue_chart | safe }}
        </div>
        {% endif %}

        {% if result.net_income_chart %}
        <div class="chart">
            {{ result.net_income_chart | safe }}
        </div>
        {% endif %}

        {% if result.ocf_chart %}
        <div class="chart">
            {{ result.ocf_chart | safe }}
        </div>
        {% endif %}
    </section>

    <section class="facts">
        <h3>SEC tags used</h3>
        <p class="small">
            These are the US-GAAP taxonomy tags selected from SEC Company Facts.
        </p>

        {% for key, value in result.facts_used.items() %}
            <div class="small">
                <strong>{{ key }}:</strong>
                <code>{{ value or 'Not found' }}</code>
            </div>
        {% endfor %}
    </section>
    {% endif %}

</div>
</body>
</html>
"""


@app.route("/", methods=["GET"])
def index():
    ticker = request.args.get("ticker", "").strip()
    result = None
    error = None

    if ticker:
        try:
            result = analyze_stock(ticker)
        except Exception as exc:
            error = str(exc)

    return render_template_string(
        HTML_TEMPLATE,
        ticker=ticker,
        result=result,
        error=error
    )


if __name__ == "__main__":
    app.run(debug=True)